## Creating Spark Session

In [1]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel
import time

spark = SparkSession.builder \
    .appName("Task4_Distributed_Computing") \
    .config("spark.driver.memory","8g") \
    .config("spark.executor.memory","8g") \
    .config("spark.sql.shuffle.partitions","16") \
    .getOrCreate()

## Loading Dataset

In [2]:
df = spark.read.parquet("nifty_10m_fixed.parquet")

## Dataset Information

In [3]:
print("Rows :", df.count())
print("Columns :", len(df.columns))

Rows : 10000000
Columns : 15


## Number of Partitions

In [4]:
print("Current Partitions :", df.rdd.getNumPartitions())

Current Partitions : 3


## Repartition Dataset

In [5]:
df = df.repartition(16)

print("Partitions after Repartition :")

print(df.rdd.getNumPartitions())

Partitions after Repartition :
16


## Cache Dataset

In [6]:
start = time.time()

df.cache()

df.count()

cache_time = time.time()-start

print("Caching Time :",round(cache_time,2),"seconds")

Caching Time : 87.58 seconds


## Persist Dataset

In [7]:
start = time.time()

df.persist(StorageLevel.MEMORY_AND_DISK)

df.count()

persist_time = time.time()-start

print("Persist Time :",round(persist_time,2),"seconds")

Persist Time : 0.47 seconds


## Comparing Cached Performance

In [8]:
df.unpersist()

start = time.time()

df.groupBy("option_type").count().show()

without_cache = time.time()-start

print("Execution Time Without Cache :",round(without_cache,2))

+-----------+-------+
|option_type|  count|
+-----------+-------+
|       CALL|5429139|
|        PUT|4570861|
+-----------+-------+

Execution Time Without Cache : 10.11


## Cache Again

In [9]:
df.cache()

df.count()

start = time.time()

df.groupBy("option_type").count().show()

with_cache = time.time()-start

print("Execution Time With Cache :",round(with_cache,2))

+-----------+-------+
|option_type|  count|
+-----------+-------+
|       CALL|5429139|
|        PUT|4570861|
+-----------+-------+

Execution Time With Cache : 2.86


## Performance Comparison

In [10]:
print("="*60)

print("WITHOUT CACHE :",round(without_cache,2),"seconds")

print("WITH CACHE :",round(with_cache,2),"seconds")

print("="*60)

WITHOUT CACHE : 10.11 seconds
WITH CACHE : 2.86 seconds


## Resource Configuration

In [11]:
print("="*50)

print("Spark Configurations")

print("="*50)

print("Driver Memory :",spark.conf.get("spark.driver.memory"))

print("Executor Memory :",spark.conf.get("spark.executor.memory"))

print("Shuffle Partitions :",spark.conf.get("spark.sql.shuffle.partitions"))

Spark Configurations
Driver Memory : 8g
Executor Memory : 8g
Shuffle Partitions : 16


## Spark UI Link

In [12]:
print(spark.sparkContext.uiWebUrl)

http://f0feed15fedd:4040
